# Working patch clamp data recorded during patterned optogenetic stimulation

In [ ]:
from mimo_pack.fileio.abf import load_abf_xr



In [24]:
abfpath = "D:\\BetaGamma\\project\\DH46\\Slice1\\Cell2\\25707017.abf"

iv = load_abf_xr(abfpath)
print(iv)

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_adcSection', '_cacheStimulusFiles', '_dacSection', '_dataGain', '_dataOffset', '_dataSection', '_dtype', '_epochPerDacSection', '_epochSection', '_fileGUID', '_fileSize', '_getAdcNameAndUnits', '_getDacNameAndUnits', '_headerV2', '_ide_helper', '_loadAndScaleData', '_makeAdditionalVariables', '_nDataFormat', '_preLoadData', '_protocolSection', '_readHeadersV1', '_readHeadersV2', '_stringsSection', '_sweepBaselinePoints', '_synchArraySection', '_tagSection', '_userListSection', 'abfDateTime', 'abfDateTimeString', 'abfFileComment', 'abfFilePath', 'abfFolderPath', 'abfID', 'abfVersion', 'abfVersionString', 'adcNames', 'adcUnits', 'channelCount', 'c

In [ ]:
import os
import glob
from PIL import Image
import numpy as np
import xarray as xr
from typing import List

def load_images_xr(directory_path: str, image_extensions: List[str] = None) -> xr.DataArray:
    """Read all images from a directory into an xarray.DataArray.

    This function scans a specified directory for image files, reads them
    using the PIL (Pillow) library, and compiles them into a single
    xarray.DataArray.

    The data is organized with dimensions ('image', 'y', 'x') for
    grayscale images or ('image', 'y', 'x', 'channel') for color images.
    The 'image' dimension is indexed numerically and includes a coordinate
    'image_name' with the corresponding filenames.

    Parameters
    ----------
    directory_path : str
        The full path to the directory containing the images.
    image_extensions : list of str, optional
        A list of image file extensions to look for (e.g., ['.png', '.jpg']).
        If None, defaults to common image formats.

    Returns
    -------
    imgs_xr: xr.DataArray
        An xarray.DataArray containing the image data. The dimensions will
        be ('image', 'y', 'x') for grayscale or 
        ('image', 'y', 'x', 'channel') for color images.

    Raises
    ------
    FileNotFoundError
        If no images with the specified extensions are found in the directory.
    ValueError
        If the images in the directory result in an unexpected number of
        array dimensions after stacking.
    """
    if image_extensions is None:
        image_extensions = ['.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp', '.gif']

    # Find all image files in the directory
    search_paths = [os.path.join(directory_path, f"*{ext}") for ext in image_extensions]
    image_files = []
    for path in search_paths:
        image_files.extend(glob.glob(path))

    if not image_files:
        raise FileNotFoundError(f"No images found in directory: {directory_path}")

    # Read all images into a list of numpy arrays using PIL
    images_list = [np.array(Image.open(f)) for f in image_files]
    
    # Get the filenames to use as a coordinate
    image_names = [os.path.basename(f) for f in image_files]

    # Stack the images into a single numpy array
    # This will create a 3D array for grayscale or 4D for color images
    all_images_data = np.stack(images_list, axis=0)
    
    # Define dimensions based on the shape of the stacked numpy array
    if all_images_data.ndim == 3: # Grayscale images
        dims = ("image", "y", "x")
        coords = {
            "image": np.arange(len(image_files)),
            "y": np.arange(all_images_data.shape[1]),
            "x": np.arange(all_images_data.shape[2]),
            "image_name": ("image", image_names),
        }
    elif all_images_data.ndim == 4: # Color images
        dims = ("image", "y", "x", "channel")
        coords = {
            "image": np.arange(len(image_files)),
            "y": np.arange(all_images_data.shape[1]),
            "x": np.arange(all_images_data.shape[2]),
            "channel": np.arange(all_images_data.shape[3]),
            "image_name": ("image", image_names),
        }
    else:
        raise ValueError(f"Unexpected number of dimensions in image data: {all_images_data.ndim}")


    # Create the xarray DataArray
    imgs_xr = xr.DataArray(
        all_images_data,
        dims=dims,
        coords=coords,
    )
    
    # Add attributes for metadata
    imgs_xr.attrs['directory_path'] = directory_path
    
    return imgs_xr